# Pretrain Commutative Transformer Encoder

Load the shared unlabeled pretraining dataset and save commutative transformer encoder weights for downstream transformer experiments.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

from src.ml import (
    CommutativeTransformerClassifier,
    CommutativeTransformerConfig,
    CommutativeTransformerPretrainingConfig,
    LossWeightConfig,
    OptimizationConfig,
    create_experiment_run,
    load_commutative_transformer_pretraining_config,
    persist_pretraining_artifacts,
    write_commutative_transformer_pretraining_config,
)
from src.tensor_utils import load_unlabeled_tensor_dataset

In [ ]:
# User inputs

unlabeled_dataset_path = Path(".dataset_cache/unlabeled_active_high_mid_low_t20_z5_y96_x96_chunks")
experiment_output_dir = Path("artifacts/pretrained_commutative_transformer")
experiment_run = create_experiment_run(experiment_output_dir, "12T_pretrain_commutative_transformer")
pretrained_encoder_path = Path(experiment_run.run_dir) / f"{experiment_run.experiment_id}_encoder_state.pt"

model_config = CommutativeTransformerConfig(
    spatial_patch_size_st=(1, 32, 32),
    spatial_patch_size_ts=(1, 32, 32),
    temporal_patch_size_ts=2,
    embed_dim=32,
    num_heads=2,
    mlp_ratio=2.0,
    dropout=0.3,
    attention_dropout=0.1,
    st_spatial_depth=1,
    st_temporal_depth=1,
    ts_temporal_depth=1,
    ts_spatial_depth=1,
    embedding_dim=16,
    probe_region_grid=(1, 2, 2),
    probe_time_bins=8,
    probe_frequency_bins=4,
)
optimization_config = OptimizationConfig(
    batch_size=8,
    epochs=90,
    learning_rate=7.5e-5,
    weight_decay=3e-3,
    early_stopping_patience=14,
    early_stopping_min_delta=0.0,
    early_stopping_start_epoch=24,
    early_stopping_monitor="loss",
    early_stopping_smoothing="median",
    early_stopping_smoothing_window=5,
    training_plot_dir=str(Path(experiment_run.loss_plot_dir) / "pretraining"),
    training_plot_every_n_epochs=1,
    training_plot_smoothing_window=5,
    scheduler_patience=5,
    scheduler_factor=0.85,
    scheduler_min_lr=1e-6,
    validation_split=0.0,
    random_state=0,
    standardize=True,
    device=None,
    verbose=True,
)
loss_weight_config = LossWeightConfig(
    lambda_cross=0.35,
    lambda_align=0.0,
    cross_warmup_epochs=12,
    cross_ramp_epochs=24,
    probe_mask_probability=0.5,
    probe_alpha_local=1.0,
    probe_alpha_region_time=1.0,
    probe_alpha_derivative=1.0,
    probe_alpha_frequency=1.0,
    probe_alpha_correlation=1.0,
)

pretraining_config = CommutativeTransformerPretrainingConfig(
    unlabeled_dataset_path=unlabeled_dataset_path,
    pretrained_encoder_path=pretrained_encoder_path,
    model_config=model_config,
    optimization_config=optimization_config,
    loss_weight_config=loss_weight_config,
)
pretraining_config_path = write_commutative_transformer_pretraining_config(pretraining_config)
pretraining_config = load_commutative_transformer_pretraining_config(pretraining_config_path)
print(f"Experiment id: {experiment_run.experiment_id}")
print(f"Experiment run folder: {Path(experiment_run.run_dir).resolve()}")
print(f"Pretraining loss PDFs: {Path(optimization_config.training_plot_dir).resolve()}")
print(f"Loaded commutative transformer pretraining config from {pretraining_config_path}")
pretraining_config

In [ ]:
unlabeled_dataset = load_unlabeled_tensor_dataset(unlabeled_dataset_path)
unlabeled_dataset["tensors"].shape, unlabeled_dataset["metadata"].shape

In [ ]:
model = CommutativeTransformerClassifier(
    model_config=model_config,
    optimization_config=optimization_config,
    loss_weight_config=loss_weight_config,
)
model.pretrain(unlabeled_dataset["tensors"])
pretrained_encoder_path = model.save_pretrained_encoder(pretrained_encoder_path)
pretraining_artifacts = persist_pretraining_artifacts(
    output_dir=experiment_output_dir,
    estimator=model,
    config=pretraining_config,
    experiment_prefix="12T_pretrain_commutative_transformer",
    experiment_id=experiment_run.experiment_id,
    pretrained_encoder_path=pretrained_encoder_path,
    loss_plot_dirs=[optimization_config.training_plot_dir],
)
pretraining_artifacts

In [ ]:
model.pretrain_history_.tail()